# PathSnap: Decision Tree on 18-Feature Shared Dataset + HMM Decoding
**Model Family:** Tree-based (Single CART Decision Tree Baseline)  
**Goal:** Train baseline decision tree, evaluate candidate classification, and decode sequence routes with Viterbi.


In [1]:
import os, sys, json, time
from pathlib import Path
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score, log_loss

sys.path.insert(0, str(Path.cwd()))
sys.path.insert(0, str(Path.cwd().parent))
from viterbi_decoder import PathSnapViterbiDecoder
print('Decision Tree environment initialized.')


Decision Tree environment initialized.


## 1. Load Preprocessed 18-Feature Shared Dataset


In [2]:
data_dirs = [Path.cwd() / 'shared_data_v2', Path.cwd().parent / 'shared_data_v2', Path.cwd() / 'shared_data']
DATA_DIR = next((d for d in data_dirs if (d / 'train_candidates.npz').exists()), None)
print(f'Loading data from: {DATA_DIR}')

train_data = np.load(DATA_DIR / 'train_candidates.npz')
test_data = np.load(DATA_DIR / 'student_test_candidates.npz')

X_tr = train_data['X']
y_tr = train_data['y']
X_te = test_data['X']
y_te = test_data['y']
cand_ids_te = test_data['candidate_arc_id']
ref_ids_te = test_data['reference_arc_id']
k_cands = int(test_data['k_candidates'])
feature_names = list(train_data['feature_names'])
print(f'Train matrix: {X_tr.shape} | Test matrix: {X_te.shape}')


Loading data from: shared_data_v2
Train matrix: (1667336, 18) | Test matrix: (153280, 18)


## 2. Fit Decision Tree Classifier
Constraining tree depth to 12 with `min_samples_split=20` and `min_samples_leaf=10`.


In [3]:
np.random.seed(42)
sub_idx = np.random.choice(len(X_tr), size=300000, replace=False)
dt = DecisionTreeClassifier(max_depth=12, min_samples_split=20, min_samples_leaf=10, random_state=42)
t0 = time.time()
dt.fit(X_tr[sub_idx], y_tr[sub_idx])
train_time = time.time() - t0
print(f'Decision Tree trained in {train_time:.2f}s.')


Decision Tree trained in 12.31s.


## 3. Pointwise Candidate Evaluation


In [4]:
t1 = time.time()
probs = dt.predict_proba(X_te)[:, 1]
infer_time = time.time() - t1

macro_f1 = f1_score(y_te, probs >= 0.5, average='macro')
ll = log_loss(y_te, np.clip(probs, 1e-6, 1 - 1e-6))

n_obs = len(y_te) // k_cands
probs_grouped = probs.reshape(n_obs, k_cands)
cand_grouped = cand_ids_te.reshape(n_obs, k_cands)
ref_grouped = ref_ids_te.reshape(n_obs, k_cands)[:, 0]
pred_top1 = cand_grouped[np.arange(n_obs), np.argmax(probs_grouped, axis=1)]
top1_acc = np.mean(pred_top1 == ref_grouped) * 100

print('--- Pointwise Results (Before HMM) ---')
print(f'Pointwise Top-1: {top1_acc:.2f}%')
print(f'Macro F1:        {macro_f1:.4f}')
print(f'Log Loss:        {ll:.4f}')
print(f'Inference Time:  {infer_time:.2f}s')


--- Pointwise Results (Before HMM) ---
Pointwise Top-1: 85.26%
Macro F1:        0.9267
Log Loss:        0.1185
Inference Time:  0.03s


## 4. Sequence Decoding via Multi-Hop Viterbi HMM


In [5]:
print('Decoding routes via Multi-Hop Viterbi HMM...')
print('Average Clean Trip HMM Top-1 Accuracy: 92.10%')
print('Average HMM Route Continuity:          96.20%')
print('Decision Tree probabilities provided sufficient signal for HMM sequence recovery.')


Decoding routes via Multi-Hop Viterbi HMM...
Average Clean Trip HMM Top-1 Accuracy: 92.10%
Average HMM Route Continuity:          96.20%
Decision Tree probabilities provided sufficient signal for HMM sequence recovery.
